In [ ]:
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
import numpy as np
import matplotlib as mpl
import pandas as pd
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.family'] = 'serif'
mpl.rcParams['font.serif'] = 'Times New Roman'
mpl.rcParams['font.size'] = 10

In [ ]:
import deepmind_lab
import matplotlib.pyplot as plt

observations = ['DEBUG.CAMERA.TOP_DOWN']
observations = ['RGB_INTERLEAVED']
env = deepmind_lab.Lab('openfield_map2_fixed_loc3', observations,
                       config={'width': '96',    # screen size, in pixels
                               'height': '72',   # screen size, in pixels
                               },  # lt_chasm option.
                       renderer='hardware')       # select renderer.
env.reset()
obs = env.observations()
# print(obs['DEBUG.CAMERA.TOP_DOWN'].dtype)
# plt.figure()
# plt.imshow(obs)
# plt.savefig('tryfig.png')

In [ ]:
import time
from collections import deque
from typing import Dict, Tuple

import gymnasium as gym
import numpy as np
import torch
from torch import Tensor

from sample_factory.algo.learning.learner import Learner
from sample_factory.algo.sampling.batched_sampling import preprocess_actions
from sample_factory.algo.utils.action_distributions import argmax_actions
from sample_factory.algo.utils.env_info import extract_env_info
from sample_factory.algo.utils.make_env import make_env_func_batched
from sample_factory.algo.utils.misc import ExperimentStatus
from sample_factory.algo.utils.rl_utils import make_dones, prepare_and_normalize_obs
from sample_factory.algo.utils.tensor_utils import unsqueeze_tensor
from sample_factory.cfg.arguments import load_from_checkpoint
from sample_factory.huggingface.huggingface_utils import generate_model_card, generate_replay_video, push_to_hf
from sample_factory.model.actor_critic import create_actor_critic
from sample_factory.model.model_utils import get_rnn_size
from sample_factory.utils.attr_dict import AttrDict
from sample_factory.utils.typing import Config, StatusCode
from sample_factory.utils.utils import debug_log_every_n, experiment_dir, log

In [ ]:
import datetime, pathlib, json,  pandas as pd, torch

def _ensure_parent(path: pathlib.Path):
    path.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
# from sample_factory.arguments import load_from_checkpoint
cfg_filename='/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNFixedSequenceLORA42_/00_RNNFixedSequenceLORA42_see_1111/config.json'
with open(cfg_filename, "r") as json_file:
    json_params = json.load(json_file)
    log.warning("Loading existing experiment configuration from %s", cfg_filename)
    loaded_cfg = AttrDict(json_params)

# # override the parameters in config file with values passed from command line
# for key, value in cfg.cli_args.items():
#     if key in loaded_cfg and loaded_cfg[key] != value:
#         log.debug("Overriding arg %r with value %r passed from command line", key, value)
#         loaded_cfg[key] = value

# # incorporate extra CLI parameters that were not present in JSON file
# for key, value in vars(cfg).items():
#     if key not in loaded_cfg:
#         log.debug("Adding new argument %r=%r that is not in the saved config file!", key, value)
#         loaded_cfg[key] = value

In [ ]:
cfg=loaded_cfg

In [ ]:
cfg['train_dir']="/work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNFixedSequenceLORA42_"
cfg['encoder_conv_architecture']='resnet'

In [ ]:
    cfg = load_from_checkpoint(cfg)

[2026-01-23 16:16:31,071][3987835] Loading existing experiment configuration from /work/classic/fr_lr554-TrainSpace/spectral_radius/train_dir/hipposlam/RNNFixedSequenceLORA42_/00_RNNFixedSequenceLORA42_see_1111/config.json


In [ ]:
REDUCED_ACTION_SET = (
    (0, 0, 0, 1, 0, 0, 0),  # Forward
    # (0, 0, 0, -1, 0, 0, 0),  # Backward
    (0, 0, -1, 0, 0, 0, 0),  # Strafe Left
    (0, 0, 1, 0, 0, 0, 0),  # Strafe Right
    # (-20, 0, 0, 0, 0, 0, 0),  # Look Left
    # (20, 0, 0, 0, 0, 0, 0),  # Look Right
    (-20, 0, 0, 1, 0, 0, 0),  # Look Left + Forward
    (20, 0, 0, 1, 0, 0, 0),  # Look Right + Forward
    # (0, 0, 0, 0, 1, 0, 0),  # Fire.
)
action_space = gym.spaces.Discrete(len(REDUCED_ACTION_SET))
observation_space = gym.spaces.Dict(
    obs=gym.spaces.Box(low=0, high=255, shape=(72, 96, 3), dtype=np.uint8)
)

In [ ]:
cfg['encoder_conv_architecture']='resnet'

In [ ]:
actor_critic = create_actor_critic(cfg,observation_space, action_space)

[2026-01-23 16:16:37,588][3987835] RunningMeanStd input shape: (72, 96, 3)
[2026-01-23 16:16:37,594][3987835] RunningMeanStd input shape: (1,)
[2026-01-23 16:16:37,611][3987835] Num input channels: 72
[2026-01-23 16:16:37,628][3987835] Convolutional layer output size: 384
[2026-01-23 16:16:37,646][3987835] get out size called: {self.core_output_size}
